# 16. SQL UDFs, Stored Procedures & Triggers: Beginner Guide

### 📝 Universal SQL Execution Order (All Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline ─────────────────────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. SELECT & CASE   ➔ 8. DISTINCT (Dedup)   ➔ 9. ORDER BY (Sorting)         │
│ ➔ 10. LIMIT / OFFSET (Final Page Slice)                                      │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **16. SQL UDFs, Stored Procedures & Triggers**. Server-side programmability allows executing procedural business logic directly within the database engine close to data storage. This notebook covers custom user-defined functions (UDFs), event-driven database triggers (`CREATE TRIGGER` with `BEFORE`/`AFTER` hooks), automated audit logging, and managing trigger side-effects safely.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 User-Defined Scalar Functions (UDFs): Extending SQL Functions
- [x] 🔹 Event-Driven Triggers: `CREATE TRIGGER ... AFTER INSERT`
- [x] 🔹 Automated Audit Logging & State History Tracking
- [x] 🔍 Scenario: Automated Real-Time Fraud Alert Trigger & Compliance Ledger Hook










In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 User-Defined Functions (UDFs)
- **What it does:** Registers custom scalar algorithms with the SQL query engine to execute reusable business logic within standard `SELECT` expressions.
- **Syntax:** Register callable function with database connection.
- **Dataset Application & Code Demonstration:** Implements and invokes a custom currency converter UDF.


In [2]:
# Register custom Python UDF in SQLite engine
def eur_to_usd(amount_eur, fx_rate=1.08):
    if amount_eur is None:
        return None
    return round(float(amount_eur) * fx_rate, 2)

conn.create_function("TO_USD", 1, eur_to_usd)
print("Registered TO_USD(amount) UDF")


Registered TO_USD(amount) UDF


In [3]:
%%sql
SELECT 
    transaction_id,
    transaction_amount AS amount_eur,
    TO_USD(transaction_amount) AS amount_usd
FROM transactions
WHERE transaction_amount IS NOT NULL
LIMIT 5;


,transaction_id,amount_eur,amount_usd
0,TX109326,607.78,656.40
1,TX106376,1819.11,1964.64
2,TX103301,64.08,69.21
3,TX110701,1025.73,1107.79
4,TX103284,772.74,834.56


### 🔹 Event-Driven Triggers: `CREATE TRIGGER ... AFTER INSERT`
- **What it does:** Automatically fires an internal SQL procedure whenever an `INSERT`, `UPDATE`, or `DELETE` mutation occurs on a target table.
- **Syntax:** `CREATE TRIGGER trigger_name AFTER INSERT ON target_table BEGIN ... END;`
- **Dataset Application & Code Demonstration:** Builds an automated audit log trigger for high-value transactions.


In [4]:
%%sql
CREATE TABLE IF NOT EXISTS high_value_audit_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    transaction_id TEXT,
    alert_amount REAL,
    logged_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TRIGGER IF NOT EXISTS trg_audit_high_value
AFTER INSERT ON transactions
WHEN NEW.transaction_amount > 1900.00
BEGIN
    INSERT INTO high_value_audit_log (transaction_id, alert_amount)
    VALUES (NEW.transaction_id, NEW.transaction_amount);
END;

INSERT INTO transactions (transaction_id, customer_id, transaction_amount, is_fraud)
VALUES ('TX_LIVE_99', 'CUST_501', 1950.00, 1);

SELECT * FROM high_value_audit_log;


'Query Executed Successfully.'

## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Trigger Performance Overhead & Anti-Patterns
- **Objective:** Analyze the performance impact of synchronous triggers on high-throughput OLTP ingestion pipelines.
- **Approach:** Compare synchronous row-by-row triggers against asynchronous outbox messaging patterns (CDC).


In [5]:
%%sql
SELECT 
    'Synchronous Triggers' AS pattern, 'In-Process (Blocking)' AS latency_impact, 'Strong ACID (Atomic)' AS consistency, 'Row-by-Row OLTP Overhead' AS risk
UNION ALL
SELECT 'Change Data Capture (CDC)', 'Asynchronous (WAL Stream)', 'Eventual Consistency', 'Near-Zero Write Overhead';


,pattern,latency_impact,consistency,risk
0,Synchronous Triggers,In-Process (Blocking),Strong ACID (Atomic),Row-by-Row OLTP Overhead
1,Change Data Capture (CDC),Asynchronous (WAL Stream),Eventual Consistency,Near-Zero Write Overhead
